# 08 · Databricks Asset Bundles: ML Assets as Code

**Goal:** understand what a bundle is, write one, and see how its YAML becomes a real job.

A **bundle** is a folder that keeps a project's code **and** its infrastructure together:

```
concepts_bundle/
├── databricks.yml              ← bundle name, targets (dev / prod), shared settings
├── resources/
│   └── train_job.yml           ← the job: tasks, dependencies, parameters
└── src/
    └── train.py                ← the notebook the job runs
```

The Databricks CLI turns the folder into real workspace objects:

| Command | What it does |
|---|---|
| `databricks bundle validate -t dev` | Checks the YAML |
| `databricks bundle deploy -t dev` | Uploads the code and creates or updates the job |
| `databricks bundle run train_job -t dev` | Runs the job |
| `databricks bundle destroy -t dev` | Removes everything the bundle created |

The CLI runs in a terminal. In this notebook we write the bundle files, then perform the **deploy** step ourselves with
the SDK, so you can see exactly how each YAML field maps onto the job.

In [ ]:
%run ./Includes/_config

In [ ]:
import base64, time, yaml
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
BUNDLE = f"{here.rsplit('/', 1)[0]}/08_concepts_bundle"

## 1. Write the bundle files

In [ ]:
databricks_yml = f"""
bundle:
  name: sws_concepts_bundle

include:
  - resources/*.yml

variables:
  table:
    description: Training table
    default: {TABLE}
  min_auc:
    default: "0.75"

targets:
  dev:
    mode: development
    default: true
  prod:
    mode: production
"""

train_job_yml = f"""
resources:
  jobs:
    train_job:
      name: "SWSAcademy - Concepts Bundle Job ({slug})"
      tags:
        course: swsacademy
      parameters:
        - name: table
          default: ${{var.table}}
        - name: min_auc
          default: ${{var.min_auc}}
      tasks:
        - task_key: train
          notebook_task:
            notebook_path: ../src/train
"""

train_py = f"""# Databricks notebook source
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
mlflow.set_experiment("{EXPERIMENT}")
pdf = spark.table(dbutils.widgets.get("table")).sample(fraction=0.3, seed=5).toPandas()
features = [c for c in pdf.columns if c not in ("Diabetes_binary", "id")]
X_tr, X_te, y_tr, y_te = train_test_split(pdf[features], pdf["Diabetes_binary"], test_size=0.25, random_state=5)
with mlflow.start_run(run_name="bundle_job_train"):
    auc = roc_auc_score(y_te, LogisticRegression(max_iter=1000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1])
    mlflow.log_metric("test_auc", auc)
print(f"AUC = {{auc:.4f}}")
assert auc >= float(dbutils.widgets.get("min_auc")), "Model below the required AUC"
"""

def put(path, text, notebook=False):
    body = {"path": path, "overwrite": True, "content": base64.b64encode(text.encode()).decode()}
    body.update({"format": "SOURCE", "language": "PYTHON"} if notebook else {"format": "AUTO"})
    w.api_client.do("POST", "/api/2.0/workspace/import", body=body)

for folder in (BUNDLE, f"{BUNDLE}/resources", f"{BUNDLE}/src"):
    w.workspace.mkdirs(folder)
put(f"{BUNDLE}/databricks.yml", databricks_yml.strip() + "\n")
put(f"{BUNDLE}/resources/train_job.yml", train_job_yml.strip() + "\n")
put(f"{BUNDLE}/src/train", train_py, notebook=True)
print("Bundle written to", BUNDLE)
print("Open it in the workspace browser to see the three files.")

## 2. "Deploy": what the CLI does with the YAML
The CLI reads `databricks.yml`, merges the `include`d resource files, replaces `${var...}` placeholders, turns relative
notebook paths into workspace paths, and creates or updates the job. The same steps, done here in Python:

In [ ]:
bundle = yaml.safe_load(databricks_yml)
job_spec = yaml.safe_load(train_job_yml)["resources"]["jobs"]["train_job"]

variables = {name: spec["default"] for name, spec in bundle["variables"].items()}
for p in job_spec["parameters"]:                                   # ${var.x} -> its default value
    if isinstance(p["default"], str) and p["default"].startswith("${var."):
        p["default"] = variables[p["default"][6:-1]]
for t in job_spec["tasks"]:                                        # ../src/train -> workspace path
    t["notebook_task"]["notebook_path"] = f"{BUNDLE}/src/train"
    t["notebook_task"]["source"] = "WORKSPACE"

existing = list(w.jobs.list(name=job_spec["name"]))
if existing:
    job_id = existing[0].job_id
    w.api_client.do("POST", "/api/2.1/jobs/reset", body={"job_id": job_id, "new_settings": job_spec})
    print("Updated job", job_spec["name"], job_id)
else:
    job_id = w.api_client.do("POST", "/api/2.1/jobs/create", body=job_spec)["job_id"]
    print("Created job", job_spec["name"], job_id)

## 3. "Run": the equivalent of `databricks bundle run train_job`

In [ ]:
run_id = w.api_client.do("POST", "/api/2.1/jobs/run-now", body={"job_id": job_id})["run_id"]
while True:
    run = w.api_client.do("GET", "/api/2.1/jobs/runs/get", query={"run_id": run_id})
    if run["state"]["life_cycle_state"] in ("TERMINATED", "INTERNAL_ERROR", "SKIPPED"):
        break
    time.sleep(15)
print("Bundle job run:", run["state"].get("result_state"))
assert run["state"].get("result_state") == "SUCCESS"

## 4. Why teams use bundles
- **One source of truth:** the job definition lives in Git next to the code, so changes are reviewed like code.
- **Same result everywhere:** `-t dev` and `-t prod` deploy the same bundle with different settings (for example a
  development copy that is paused and prefixed with your name in dev, and the real schedule in prod).
- **Clean up in one step:** `databricks bundle destroy` removes everything the bundle created.

**Try the real CLI** (from your laptop's terminal, after `databricks auth login`):

```bash
databricks bundle init default-python      # create a new bundle from a template
databricks bundle validate -t dev
databricks bundle deploy -t dev
databricks bundle run <job_key> -t dev
databricks bundle destroy -t dev
```

## Key points
- A bundle = code + YAML describing jobs and pipelines, kept together in Git.
- `deploy` turns the YAML into real jobs; `run` starts them; `destroy` removes them.
- This is the last step of the progression from Folder 4: jobs built in the UI → with the API → with the CLI → as bundles.